# LeetCode 356 — Line Reflection bằng Python

Link đề: https://leetcode.com/problems/line-reflection/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Geometry (hình học tọa độ), Sorting / Two Pointers

> **Bài Premium (🔒).** Đề trong notebook này được viết theo hiểu biết của mình về bài và theo bản mô tả đã có; nếu bạn có tài khoản Premium, hãy **đối chiếu lại đề gốc** (nhất là cách đọc "có điểm trùng lặp" và ví dụ) trước khi nộp.

Skeleton LeetCode:

```python
class Solution:
    def isReflected(self, points: List[List[int]]) -> bool:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu năm cách làm với một oracle vét cạn độc lập trên hàng nghìn bộ điểm ngẫu nhiên). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho `n` điểm trên mặt phẳng, hỏi có tồn tại **một đường thẳng song song trục `y`** (tức đường thẳng đứng `x = m`) sao cho **phản xạ (lấy đối xứng) tất cả điểm qua đường đó** thì **tập điểm** thu được **trùng với tập điểm ban đầu** hay không (điểm có thể lặp).
2. Nắm các công cụ Python: **tuple làm khóa trong `set`** (list thì không được), `min`/`max`, `all(...)` với generator, `defaultdict(list)` để **nhóm điểm theo `y`**, `sorted`, **hai con trỏ**, và mẹo **dùng số nguyên `s = 2m`** để tránh số thực.
3. Đi từ cách **thử mọi trục có thể** (vét cạn) → **trục chỉ có thể là trung điểm của điểm đầu với một điểm nào đó** → **trục buộc phải là `(min_x + max_x) / 2`** (hash `O(n)`) → hai cách **không cần hash tuple**: **nhóm theo `y` + hai con trỏ** và **chia trái/phải + sort + ghép cặp**.
4. Hiểu mấu chốt: điểm có `x` nhỏ nhất **buộc** phải phản xạ thành điểm có `x` lớn nhất, nên trục duy nhất có thể là `x = (min_x + max_x) / 2`; chỉ cần **kiểm tra một lần** cho mọi điểm: `(min_x + max_x − x, y)` có nằm trong tập không.
5. Tránh các lỗi kinh điển: chỉ xét `x` mà **quên `y`**, chia nguyên `//` làm sai trục nửa nguyên, dùng **trung bình** `x` thay vì `min + max`, bỏ vào `set` một `list` (unhashable), và đếm điểm trùng như điểm riêng biệt.

## 2. Đề bài & ví dụ

Cho `n` điểm trên mặt phẳng 2D. Hãy xác định xem có tồn tại một **đường thẳng song song với trục `y`** sao cho **phản xạ** (đối xứng) các điểm đã cho qua đường thẳng đó thì **tập điểm sau khi phản xạ giống hệt tập điểm ban đầu** hay không.

Nói cách khác: có đường thẳng `x = m` nào mà với **mọi** điểm `(x, y)` đã cho, điểm đối xứng `(2m − x, y)` **cũng** là một trong các điểm đã cho hay không.

**Lưu ý:** có thể có các điểm **lặp lại** (trùng nhau).

**Ví dụ 1**

- Input: `points = [[1,1],[-1,1]]`
- Output: `true`
- Giải thích: chọn đường thẳng `x = 0`.

**Ví dụ 2**

- Input: `points = [[1,1],[-1,-1]]`
- Output: `false`
- Giải thích: không có đường thẳng nào thỏa (điểm `(1,1)` cần `(−1,1)` ở đối diện, nhưng bộ điểm không có).

**Ràng buộc:**

- `n == points.length`
- `1 <= n <= 10^4`
- `-10^8 <= points[i][j] <= 10^8`

**Follow-up:** bạn có làm tốt hơn `O(n²)` được không?

In [1]:
import random
from collections import Counter, defaultdict
from typing import List

examples = [
    ([[1, 1], [-1, 1]], True),
    ([[1, 1], [-1, -1]], False),
]
for pts, expected in examples:
    print(pts, "-> kỳ vọng", expected)

[[1, 1], [-1, 1]] -> kỳ vọng True
[[1, 1], [-1, -1]] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Phản xạ qua đường thẳng đứng `x = m`

Đường thẳng đứng `x = m` song song trục `y`. Phản xạ điểm `(x, y)` qua nó cho điểm có **cùng độ cao `y`** và khoảng cách tới đường là như nhau nhưng ở **phía bên kia**: `(2m − x, y)`. Ví dụ `m = 0`: `(1, 1) → (−1, 1)`; `m = 2`: `(5, 3) → (−1, 3)`; `m = 0.5`: `(0, 7) → (1, 7)`.

Để **tránh số thực**, đặt `s = 2m` (một số **nguyên** nếu `m` nguyên hoặc nửa nguyên). Khi đó phản xạ là `(s − x, y)`: chỉ cần cộng/trừ số nguyên.

In [2]:
def reflect(point, s):
    """Phản xạ điểm qua đường thẳng x = s/2 (s = 2m là số nguyên)."""
    x, y = point
    return (s - x, y)

print(reflect((1, 1), 0))      # (-1, 1): trục x = 0
print(reflect((5, 3), 4))      # (-1, 3): trục x = 2
print(reflect((0, 7), 1))      # (1, 7): trục x = 0.5 (s = 1, nửa nguyên)
print(reflect(reflect((5, 3), 4), 4))   # (5, 3): phản xạ hai lần = điểm ban đầu

(-1, 1)
(-1, 3)
(1, 7)
(5, 3)


### 3.2. `set` và `tuple` — vì sao list không bỏ vào `set` được?

`set` cho phép kiểm tra `p in S` trong `O(1)` trung bình, nhưng phần tử phải **hashable (băm được)**. `tuple` bất biến nên băm được; `list` có thể thay đổi nên **không**. Vậy cần đổi mỗi điểm `[x, y]` thành `(x, y)`. Đưa các điểm vào `set` cũng **loại điểm trùng lặp** — đúng với yêu cầu "tập điểm".

In [3]:
points = [[0, 0], [1, 5], [0, 0]]          # điểm (0,0) lặp
try:
    bad = {p for p in points}              # p là list -> không băm được
except TypeError as e:
    print("lỗi:", e)

S = {(x, y) for x, y in points}            # đổi sang tuple
print(S, len(S), "điểm khác nhau")
print((1, 5) in S, (5, 1) in S)            # True False

lỗi: unhashable type: 'list'
{(1, 5), (0, 0)} 2 điểm khác nhau
True False


### 3.3. `min`, `max`, `all`

- `min(x for x, y in points)` / `max(...)`: tọa độ `x` nhỏ nhất / lớn nhất.
- `all(điều_kiện for ...)`: `True` nếu **mọi** phần tử thỏa; dừng sớm ngay khi gặp phần tử đầu tiên không thỏa. Với dãy rỗng trả `True`.

In [4]:
points = [[1, 1], [-1, 1], [3, 0]]
mn = min(x for x, y in points)
mx = max(x for x, y in points)
print(mn, mx, mn + mx)
print(all(x > -5 for x, y in points), all(x > 0 for x, y in points), all(False for _ in []))

-1 3 2
True False True


### 3.4. Nhóm điểm theo `y` bằng `defaultdict(list)`

Phản xạ qua đường **thẳng đứng** giữ nguyên `y`. Vì thế các điểm có cùng `y` chỉ "nói chuyện" với nhau: ta có thể nhóm theo `y` và xét từng **hàng ngang** một.

In [5]:
points = [[1, 1], [-1, 1], [3, 0], [2, 1]]
rows = defaultdict(list)
for x, y in points:
    rows[y].append(x)
print(dict(rows))                      # {1: [1, -1, 2], 3...} : y -> các x
print({y: sorted(set(xs)) for y, xs in rows.items()})

{1: [1, -1, 2], 0: [3]}
{1: [-1, 1, 2], 0: [3]}


### 3.5. Sắp xếp và hai con trỏ

Với dãy đã sắp tăng `a`, cặp "đối xứng" là `(a[0], a[-1])`, `(a[1], a[-2])`, ... — dùng hai chỉ số `i` (từ trái) và `j` (từ phải), mỗi vòng `i += 1`, `j -= 1`, dừng khi `i >= j`. Nếu dãy đối xứng qua một số `c` thì **mọi** cặp có cùng tổng `a[i] + a[j] = 2c`.

In [6]:
a = [-3, -1, 1, 3]                    # đối xứng qua 0
i, j = 0, len(a) - 1
while i < j:
    print(f"cặp ({a[i]}, {a[j]}) tổng {a[i] + a[j]}")
    i += 1
    j -= 1

a = [-3, -1, 2, 4]                    # các cặp (đầu, cuối) cùng tổng 1 -> đối xứng qua 0.5
i, j = 0, len(a) - 1
print([a[k] + a[len(a) - 1 - k] for k in range(len(a) // 2)])   # [1, 1]: cùng tổng -> trục x = 0.5

cặp (-3, 3) tổng 0
cặp (-1, 1) tổng 0
[1, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ 1:** `[[1,1],[-1,1]]`. Hai điểm cùng độ cao `y = 1`, một ở `x = 1`, một ở `x = −1`. Trục phải nằm **chính giữa** hai điểm: `m = (1 + (−1)) / 2 = 0`.

**Ví dụ 2:** `[[1,1],[-1,-1]]`. Điểm `x` nhỏ nhất là `−1`, lớn nhất là `1`, trục nếu có thì là `x = 0` (`s = 0`). Phản xạ `(1,1)` → `(−1,1)`: có trong bộ điểm không? Không (bộ chỉ có `(−1,−1)`). ⇒ `false`.

**Ví dụ tự dựng:** `[[0,0],[4,0],[1,3],[3,3]]`.

| điểm `(x,y)` | điểm phản xạ `(s−x, y)` với `s = min+max = 0+4 = 4` | có trong bộ? |
|---|---|---|
| `(0,0)` | `(4,0)` | có |
| `(4,0)` | `(0,0)` | có |
| `(1,3)` | `(3,3)` | có |
| `(3,3)` | `(1,3)` | có |

⇒ `true`.

**Quan sát then chốt.** Giả sử có trục `x = m` thỏa. Điểm có `x` **nhỏ nhất** (`min_x`) phản xạ thành điểm có `x = 2m − min_x`, điểm này cũng phải thuộc bộ điểm nên `2m − min_x <= max_x`. Tương tự điểm có `x` lớn nhất phản xạ thành `2m − max_x >= min_x`. Suy ra `2m <= min_x + max_x` và `2m >= min_x + max_x`, tức **`2m = min_x + max_x`**. Vậy **chỉ có đúng một trục để thử**.

In [7]:
points = [[0, 0], [4, 0], [1, 3], [3, 3]]
S = {tuple(p) for p in points}
s = min(p[0] for p in points) + max(p[0] for p in points)
print("s = min + max =", s, "=> trục x =", s / 2)
for x, y in points:
    mirror = (s - x, y)
    print((x, y), "->", mirror, "có trong bộ?", mirror in S)

s = min + max = 4 => trục x = 2.0
(0, 0) -> (4, 0) có trong bộ? True
(4, 0) -> (0, 0) có trong bộ? True
(1, 3) -> (3, 3) có trong bộ? True
(3, 3) -> (1, 3) có trong bộ? True


## 5. Cách 1 — Brute Force / baseline: thử mọi trục từ mọi cặp `x`

**Ý tưởng.** Chưa biết trục ở đâu, ta thử mọi khả năng "hợp lý": trục phải là trung điểm của hai tọa độ `x` nào đó (nếu có điểm không nằm trên trục thì nó cần một bạn đối xứng; còn nếu mọi điểm nằm đúng trên trục thì `x` của chúng bằng nhau = trục chính là `x` đó, tức cặp `(x, x)` cũng nằm trong các cặp). Với mỗi `s = x_i + x_j` (mọi cặp `i <= j`), kiểm tra: **mọi điểm** `(x, y)` có `(s − x, y)` trong tập không?

In [8]:
def is_reflected_all_pairs(points):
    pts = {(x, y) for x, y in points}
    xs = sorted({x for x, y in points})
    tried = set()
    for i in range(len(xs)):
        for j in range(i, len(xs)):
            s = xs[i] + xs[j]
            if s in tried:
                continue
            tried.add(s)
            if all((s - x, y) in pts for x, y in pts):      # kiểm tra mọi điểm với trục này
                return True
    return False


for pts, expected in examples:
    print(is_reflected_all_pairs(pts), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Có tới `O(k²)` giá trị `s` (với `k` = số `x` khác nhau, tối đa `n`), mỗi giá trị kiểm tra `O(n)` ⇒ **Time `O(n³)`** tệ nhất; **Space `O(n)`**. Với `n = 10^4` hoàn toàn không dùng được.

**Nút thắt:** thử quá nhiều trục. Ta cần **thu hẹp** số trục ứng viên.

### Thu hẹp bớt: trục chỉ có thể xuất phát từ điểm ĐẦU TIÊN

Điểm `p0 = points[0]` bắt buộc phải phản xạ thành một điểm `q` **thuộc bộ điểm** và `q` cùng `y` với `p0`. Vậy `s = x0 + x_q` với `q` là một điểm có `y = y0`. Chỉ cần thử `s` ứng với các điểm cùng hàng với `p0` — tối đa `n` giá trị `s`.

In [9]:
def is_reflected_first_point(points):
    pts = {(x, y) for x, y in points}
    x0, y0 = points[0]
    for x, y in pts:
        if y != y0:
            continue
        s = x0 + x                                          # p0 phản xạ thành (x, y0)
        if all((s - px, py) in pts for px, py in pts):      # kiểm tra mọi điểm
            return True
    return False


for pts, expected in examples:
    print(is_reflected_first_point(pts), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp (thu hẹp):** tối đa `n` ứng viên `s`, mỗi ứng viên kiểm tra `O(n)` ⇒ **Time `O(n²)`**, **Space `O(n)`**. Đây đúng là "cách `O(n²)`" mà phần follow-up của đề gợi ý phải làm tốt hơn. Nhưng bước đột phá bên dưới cho thấy chỉ còn **một** ứng viên.

## 6. Ý tưởng thuật toán / bước đột phá

Từ phần tính tay ở mục 4: nếu tồn tại trục `x = m`, thì **bắt buộc** `2m = min_x + max_x`. Đặt `s = min_x + max_x` (số nguyên — không cần chia `2`!). Khi đó:

> Trả lời **có** ⇔ với **mọi** điểm `(x, y)`, điểm `(s − x, y)` **cũng** là một điểm trong bộ.

Chiều "⇒" đã chứng minh ở mục 4. Chiều "⇐": nếu mọi điểm đều có "bạn" `(s − x, y)` trong bộ thì phản xạ qua `x = s/2` đưa bộ vào chính nó; vì phản xạ là **song ánh** (hai lần phản xạ trả về ban đầu) nên ảnh **đúng bằng** bộ điểm (không thể "nhỏ hơn" vì ảnh của ảnh quay về bộ). ∎

Vậy chỉ cần: **một lần duyệt** lấy `min_x`, `max_x` và đưa điểm vào `set`; **một lần duyệt** kiểm tra. **Time `O(n)`**.

## 7. Các cách tiếp theo

### Cách 3 — `min + max` và `set` các điểm (cách chuẩn, `O(n)`)

**Giải thích từng đoạn.**

1. Duyệt một lần: cập nhật `min_x`, `max_x`; thêm `(x, y)` vào `point_set` (tuple, nên băm được; điểm lặp tự bị gộp).
2. `s = min_x + max_x` (nguyên).
3. `all((s - x, y) in point_set for x, y in points)`: mọi điểm phải có bạn đối xứng. Trả `True`/`False`.

Điểm **nằm trên trục** (`2x = s`) có `(s − x, y) = (x, y)`: chính nó nên luôn thỏa — không cần xử lý riêng.

In [10]:
def is_reflected_minmax(points):
    min_x, max_x = float("inf"), float("-inf")
    point_set = set()
    for x, y in points:
        min_x = min(min_x, x)
        max_x = max(max_x, x)
        point_set.add((x, y))
    s = min_x + max_x
    return all((s - x, y) in point_set for x, y in points)


for pts, expected in examples:
    print(is_reflected_minmax(pts), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


In [11]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionMinMax:
    def isReflected(self, points: List[List[int]]) -> bool:
        min_x, max_x = float("inf"), float("-inf")
        point_set = set()
        for x, y in points:
            min_x = min(min_x, x)
            max_x = max(max_x, x)
            point_set.add((x, y))
        s = min_x + max_x
        return all((s - x, y) in point_set for x, y in points)


print(SolutionMinMax().isReflected([[1, 1], [-1, 1]]))     # True

True


**Độ phức tạp Cách 3:** hai lần duyệt `n` điểm, mỗi thao tác `set` `O(1)` trung bình ⇒ **Time `O(n)`**, **Space `O(n)`**.

### Cách 4 — Nhóm theo `y`, sắp xếp từng hàng và hai con trỏ (không cần hash tuple)

**Ý tưởng.** Phản xạ qua đường thẳng đứng không đổi `y`, nên các điểm cùng `y` tạo thành một **hàng** độc lập. Với mỗi hàng, lấy tập `x` khác nhau và **sắp tăng**. Hàng đó đối xứng qua `x = s/2` ⇔ các cặp ở hai đầu có cùng tổng `s`: `xs[i] + xs[−1−i] = s` với mọi `i`. Và **mọi hàng phải dùng cùng một `s`**; ta lấy `s` từ hàng đầu tiên rồi kiểm tra các hàng còn lại (dùng chung `s`).

**Giải thích từng đoạn.**

1. `rows[y]` = tập `x` của hàng `y` (dùng `set` để bỏ trùng), rồi `sorted`.
2. Lấy `s` = tổng hai đầu của hàng đầu tiên (`xs[0] + xs[-1]`).
3. Với mỗi hàng: hai con trỏ `i = 0`, `j = len − 1`; trong khi `i <= j`: nếu `xs[i] + xs[j] != s` ⇒ `False`. (Khi `i == j`: phần tử giữa: `2·xs[i] = s` nghĩa là nằm đúng trên trục; điều kiện tự kiểm tra.)
4. Mọi hàng qua ⇒ `True`.

Lưu ý: dùng `s` của **hàng đầu** thay cho `min_x + max_x` toàn cục vẫn đúng vì hàng đầu tiên cũng buộc `s` bằng tổng hai đầu của nó; nếu `s` toàn cục khác thì hàng nào đó sẽ bị `False`.

In [12]:
def is_reflected_rows(points):
    rows = defaultdict(set)
    for x, y in points:
        rows[y].add(x)                       # hàng y: tập các x (tự bỏ trùng)
    s = None
    for y, xs_set in rows.items():
        xs = sorted(xs_set)
        if s is None:
            s = xs[0] + xs[-1]               # trục suy ra từ hàng đầu tiên
        i, j = 0, len(xs) - 1
        while i <= j:                        # i == j: phần tử giữa phải nằm đúng trên trục
            if xs[i] + xs[j] != s:
                return False
            i += 1
            j -= 1
    return True


for pts, expected in examples:
    print(is_reflected_rows(pts), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionRows:
    def isReflected(self, points: List[List[int]]) -> bool:
        rows = defaultdict(set)
        for x, y in points:
            rows[y].add(x)
        s = None
        for xs_set in rows.values():
            xs = sorted(xs_set)
            if s is None:
                s = xs[0] + xs[-1]
            i, j = 0, len(xs) - 1
            while i <= j:
                if xs[i] + xs[j] != s:
                    return False
                i += 1
                j -= 1
        return True


print(SolutionRows().isReflected([[1, 1], [-1, -1]]))      # False

False


**Độ phức tạp Cách 4:** gom nhóm `O(n)`, sắp xếp các hàng tổng cộng `O(n log n)`, hai con trỏ `O(n)` ⇒ **Time `O(n log n)`**, **Space `O(n)`**. Chậm hơn Cách 3 một hệ số `log`, nhưng chỉ dùng số nguyên và `sorted`, không phụ thuộc băm tuple.

### Cách 5 — Chia trái/phải theo trục, sắp xếp rồi ghép cặp

**Ý tưởng.** Biết `s = min_x + max_x` (đã chứng minh). Loại trùng điểm. Chia điểm thành: **trái** (`2x < s`), **phải** (`2x > s`), và **trên trục** (`2x = s`; luôn tự đối xứng, bỏ qua). Phản xạ phải **ánh xạ một-một** trái ↔ phải. Sắp:

- `left` theo `(x, y)` **tăng** (xa trục nhất đầu tiên: `x` nhỏ nhất);
- `right` theo `(−x, y)` (xa trục nhất đầu tiên: `x` lớn nhất; cùng `x` thì `y` tăng).

Thế thì phần tử thứ `k` của `left` phải **đối xứng đúng** phần tử thứ `k` của `right`: `(s − x_l, y_l) == right[k]`. Số phần tử hai bên phải bằng nhau.

**Giải thích từng đoạn.**

1. `uniq = sorted(set(map(tuple, points)))` — các điểm khác nhau.
2. `left = [(x,y) ...  2*x < s]` đã đúng thứ tự `(x, y)` vì `uniq` đã sắp.
3. `right = sorted([... 2*x > s], key=lambda p: (-p[0], p[1]))`.
4. So sánh độ dài; rồi `all(...)` ghép cặp từng phần tử.

In [14]:
def is_reflected_sorted_pairs(points):
    uniq = sorted({(x, y) for x, y in points})                 # điểm khác nhau, sắp theo (x, y)
    s = uniq[0][0] + uniq[-1][0]                                # min_x + max_x (uniq sắp theo x tăng)
    left = [p for p in uniq if 2 * p[0] < s]
    right = sorted((p for p in uniq if 2 * p[0] > s), key=lambda p: (-p[0], p[1]))
    if len(left) != len(right):
        return False
    return all((s - lx, ly) == r for (lx, ly), r in zip(left, right))


for pts, expected in examples:
    print(is_reflected_sorted_pairs(pts), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )


In [15]:
# Code nộp LeetCode cho Cách 5
class SolutionSortedPairs:
    def isReflected(self, points: List[List[int]]) -> bool:
        uniq = sorted({(x, y) for x, y in points})
        s = uniq[0][0] + uniq[-1][0]
        left = [p for p in uniq if 2 * p[0] < s]
        right = sorted((p for p in uniq if 2 * p[0] > s), key=lambda p: (-p[0], p[1]))
        if len(left) != len(right):
            return False
        return all((s - lx, ly) == r for (lx, ly), r in zip(left, right))


print(SolutionSortedPairs().isReflected([[0, 0], [4, 0], [1, 3], [3, 3]]))   # True

True


**Độ phức tạp Cách 5:** sắp xếp `O(n log n)`, các bước còn lại `O(n)` ⇒ **Time `O(n log n)`**, **Space `O(n)`**. Cách này không cần băm điểm để tra cứu — thay bằng "ghép cặp theo thứ tự sau khi sắp". Điểm trên trục tự đối xứng nên bỏ qua; mọi điểm còn lại ghép trái-phải đúng một-một.

## 8. TRACE từng bước

### 8.1. Trace Cách 3 (`min + max` + `set`) trên điểm tự dựng

Điểm: `[[0,0],[4,0],[1,3],[3,3],[2,9]]` (có điểm `(2,9)` nằm đúng trên trục `x = 2`).

| bước | điểm `(x,y)` | `min_x` | `max_x` | `point_set` thêm |
|---|---|---|---|---|
| 1 | `(0,0)` | 0 | 0 | `(0,0)` |
| 2 | `(4,0)` | 0 | 4 | `(4,0)` |
| 3 | `(1,3)` | 0 | 4 | `(1,3)` |
| 4 | `(3,3)` | 0 | 4 | `(3,3)` |
| 5 | `(2,9)` | 0 | 4 | `(2,9)` |

Rồi `s = 0 + 4 = 4`; kiểm tra: `(0,0)→(4,0)` ✓, `(4,0)→(0,0)` ✓, `(1,3)→(3,3)` ✓, `(3,3)→(1,3)` ✓, `(2,9)→(2,9)` ✓ ⇒ `True`.

In [16]:
def trace_minmax(points):
    print("điểm:", points)
    min_x, max_x = float("inf"), float("-inf")
    point_set = set()
    for k, (x, y) in enumerate(points, 1):
        min_x = min(min_x, x)
        max_x = max(max_x, x)
        point_set.add((x, y))
        print(f"  bước {k}: điểm {(x, y)} -> min_x = {min_x}, max_x = {max_x}, |tập| = {len(point_set)}")
    s = min_x + max_x
    print(f"s = min_x + max_x = {s} (trục x = {s / 2})")
    ok = True
    for x, y in points:
        mirror = (s - x, y)
        found = mirror in point_set
        print(f"  {(x, y)} -> {mirror}: {'có' if found else 'KHÔNG có'}")
        if not found:
            ok = False
            break
    print("kết quả:", ok)
    return ok


assert trace_minmax([[0, 0], [4, 0], [1, 3], [3, 3], [2, 9]]) is True
assert trace_minmax([[0, 0], [4, 0], [1, 3], [3, 2]]) is False

điểm: [[0, 0], [4, 0], [1, 3], [3, 3], [2, 9]]
  bước 1: điểm (0, 0) -> min_x = 0, max_x = 0, |tập| = 1
  bước 2: điểm (4, 0) -> min_x = 0, max_x = 4, |tập| = 2
  bước 3: điểm (1, 3) -> min_x = 0, max_x = 4, |tập| = 3
  bước 4: điểm (3, 3) -> min_x = 0, max_x = 4, |tập| = 4
  bước 5: điểm (2, 9) -> min_x = 0, max_x = 4, |tập| = 5
s = min_x + max_x = 4 (trục x = 2.0)
  (0, 0) -> (4, 0): có
  (4, 0) -> (0, 0): có
  (1, 3) -> (3, 3): có
  (3, 3) -> (1, 3): có
  (2, 9) -> (2, 9): có
kết quả: True
điểm: [[0, 0], [4, 0], [1, 3], [3, 2]]
  bước 1: điểm (0, 0) -> min_x = 0, max_x = 0, |tập| = 1
  bước 2: điểm (4, 0) -> min_x = 0, max_x = 4, |tập| = 2
  bước 3: điểm (1, 3) -> min_x = 0, max_x = 4, |tập| = 3
  bước 4: điểm (3, 2) -> min_x = 0, max_x = 4, |tập| = 4
s = min_x + max_x = 4 (trục x = 2.0)
  (0, 0) -> (4, 0): có
  (4, 0) -> (0, 0): có
  (1, 3) -> (3, 3): KHÔNG có
kết quả: False


### 8.2. Trace Cách 4 (nhóm theo `y` + hai con trỏ)

Điểm: `[[0,0],[4,0],[1,3],[3,3],[2,3]]`. Hàng `y = 0`: `x = [0, 4]`; hàng `y = 3`: `x = [1, 2, 3]`.

| hàng `y` | `xs` đã sắp | `s` | `i, j` | `xs[i] + xs[j]` | kết quả |
|---|---|---|---|---|---|
| 0 | `[0, 4]` | `0 + 4 = 4` | `0, 1` | `4` | khớp |
| 3 | `[1, 2, 3]` | 4 | `0, 2` | `4` | khớp |
| 3 | | 4 | `1, 1` | `2 + 2 = 4` | khớp (phần tử giữa trên trục) |

⇒ `True`. Hàm trace dưới đây in đúng các bước.

In [17]:
def trace_rows(points):
    rows = defaultdict(set)
    for x, y in points:
        rows[y].add(x)
    print("các hàng:", {y: sorted(v) for y, v in rows.items()})
    s = None
    for y, xs_set in rows.items():
        xs = sorted(xs_set)
        if s is None:
            s = xs[0] + xs[-1]
            print(f"s lấy từ hàng đầu y={y}: {xs[0]} + {xs[-1]} = {s}")
        i, j = 0, len(xs) - 1
        while i <= j:
            total = xs[i] + xs[j]
            print(f"  hàng y={y}: i={i}, j={j}: {xs[i]} + {xs[j]} = {total} {'==' if total == s else '!='} {s}")
            if total != s:
                print("  -> False")
                return False
            i += 1
            j -= 1
    print("-> True")
    return True


assert trace_rows([[0, 0], [4, 0], [1, 3], [3, 3], [2, 3]]) is True
assert trace_rows([[0, 0], [4, 0], [1, 3], [3, 3], [2, 3], [5, 3]]) is False

các hàng: {0: [0, 4], 3: [1, 2, 3]}
s lấy từ hàng đầu y=0: 0 + 4 = 4
  hàng y=0: i=0, j=1: 0 + 4 = 4 == 4
  hàng y=3: i=0, j=2: 1 + 3 = 4 == 4
  hàng y=3: i=1, j=1: 2 + 2 = 4 == 4
-> True
các hàng: {0: [0, 4], 3: [1, 2, 3, 5]}
s lấy từ hàng đầu y=0: 0 + 4 = 4
  hàng y=0: i=0, j=1: 0 + 4 = 4 == 4
  hàng y=3: i=0, j=3: 1 + 5 = 6 != 4
  -> False


## 9. Code nộp LeetCode

Mình nộp **Cách 3** (`min + max` với `set`): `O(n)`, ngắn gọn, đúng yêu cầu follow-up.

In [18]:
from typing import List


class Solution:
    def isReflected(self, points: List[List[int]]) -> bool:
        min_x, max_x = float("inf"), float("-inf")
        point_set = set()
        for x, y in points:
            min_x = min(min_x, x)
            max_x = max(max_x, x)
            point_set.add((x, y))       # tuple để băm được; điểm lặp tự gộp
        s = min_x + max_x               # s = 2 * (hoành độ trục), luôn là số nguyên
        return all((s - x, y) in point_set for x, y in points)


sol = Solution()
for pts, expected in examples:
    print(pts, "->", sol.isReflected(pts), "(kỳ vọng", expected, ")")

[[1, 1], [-1, 1]] -> True (kỳ vọng True )
[[1, 1], [-1, -1]] -> False (kỳ vọng False )


## 10. Vì sao thuật toán đúng

- **Trục (nếu có) buộc phải là `s/2` với `s = min_x + max_x`:** nếu `x = m` là trục thỏa, điểm có `x = min_x` phản xạ thành điểm trong bộ nên `2m − min_x <= max_x`; điểm có `x = max_x` phản xạ thành điểm trong bộ nên `2m − max_x >= min_x`. Cộng lại `2m = min_x + max_x`.
- **Điều kiện cần:** nếu `Solution` trả `False` thì có điểm `(x, y)` với `(s − x, y)` không thuộc bộ; mà trục duy nhất khả dĩ là `s/2` ⇒ nó phản xạ `(x, y)` ra ngoài bộ ⇒ không tồn tại trục thỏa. Đúng.
- **Điều kiện đủ:** nếu mọi điểm có bạn đối xứng trong bộ thì phản xạ qua `s/2` đưa bộ `S` vào `S`; vì phản xạ là song ánh và là **phép đối hợp** (`f(f(p)) = p`), ảnh `f(S)` thỏa `f(S) ⊆ S` và `S = f(f(S)) ⊆ f(S)` ⇒ `f(S) = S`. Đúng.
- **Điểm lặp:** `set` gộp điểm trùng, và "tập điểm" trong đề cũng không phân biệt số lần xuất hiện, nên lặp không ảnh hưởng.
- **Điểm nằm trên trục:** `s − x = x` nên chính nó là bạn đối xứng — luôn thỏa.
- **Không dùng số thực:** `s` nguyên, `s − x` nguyên — không sợ sai số làm tròn.

## 11. Phân tích Time / Space Complexity

Gọi `n` = số điểm, `k` = số giá trị `x` khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1a. Thử mọi cặp `x` | `O(k² · n)` ≤ `O(n³)` | `O(n)` |
| 1b. Thử trục từ điểm đầu | `O(n²)` | `O(n)` |
| 3. `min + max` + `set` | **`O(n)`** | `O(n)` |
| 4. Nhóm theo `y`, hai con trỏ | `O(n log n)` | `O(n)` |
| 5. Chia trái/phải + sort + ghép cặp | `O(n log n)` | `O(n)` |

Với `n <= 10^4`, mọi cách `O(n log n)` hay `O(n)` đều tức thì; chỉ cách `O(n²)`/`O(n³)` mới chậm (`10^8` ... `10^12` bước).

## 12. Bộ test

Dựng **oracle** độc lập: `oracle_reflect(points)` thử **mọi** `s` nguyên trong khoảng bao quanh (không dùng nhận xét `s = min + max`): với mỗi `s`, kiểm tra `{(s − x, y)} == tập điểm`. Dùng được cho tọa độ nhỏ (`|x|, |y| <= R`), vì mọi trục khả dĩ có `s = 2m` nằm trong `[−2R, 2R]` — (trục nằm ngoài `[min_x, max_x]` thì ảnh nằm hoàn toàn ngoài bộ điểm).

In [19]:
def oracle_reflect(points, R):
    S = {(x, y) for x, y in points}
    for s in range(-2 * R - 2, 2 * R + 3):                  # thử mọi s nguyên (m = s/2 có thể nửa nguyên)
        if {(s - x, y) for x, y in S} == S:
            return True
    return False


METHODS = [is_reflected_all_pairs, is_reflected_first_point, is_reflected_minmax,
           is_reflected_rows, is_reflected_sorted_pairs, lambda p: Solution().isReflected(p)]


def check_points(points, expected, R=None):
    for f in METHODS:
        assert f(points) == expected, (getattr(f, "__name__", "Solution"), points, expected)
    if R is not None:
        assert oracle_reflect(points, R) == expected, ("oracle", points)


# Ví dụ đề
check_points([[1, 1], [-1, 1]], True, 1)
check_points([[1, 1], [-1, -1]], False, 1)

# Biên
check_points([[5, 7]], True, 7)                                   # một điểm: trục đi qua nó
check_points([[0, 0], [0, 0], [0, 0]], True, 0)                  # toàn điểm lặp
check_points([[3, 1], [3, 2], [3, -5]], True, 5)                  # cùng x: trục x = 3
check_points([[0, 0], [1, 0]], True, 1)                           # trục nửa nguyên x = 0.5
check_points([[0, 0], [2, 0], [1, 5]], True, 5)                   # có điểm nằm trên trục x = 1
check_points([[0, 0], [2, 0], [1, 5], [1, 6]], True, 6)           # nhiều điểm trên trục
check_points([[-100000000, 0], [100000000, 0]], True)             # biên ±10^8 (oracle bỏ qua: R quá lớn)
check_points([[-100000000, 0], [100000000, 1]], False)
check_points([[0, 0], [0, 0], [1, 0]], True, 1)                   # điểm lặp: tập {(0,0),(1,0)} đối xứng qua 0.5

# Hai hướng đều thất bại
check_points([[0, 0], [2, 1]], False, 2)                          # y khác nhau
check_points([[0, 0], [0, 1], [2, 0]], False, 2)                  # thiếu (2,1)
check_points([[0, 0], [2, 0], [3, 0]], False, 3)                  # không đối xứng
print("test cơ bản qua")

test cơ bản qua


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách sai phổ biến: `bad_only_x` (quên `y`), `bad_floor_div` (dùng `//2`), `bad_mean` (lấy trung bình `x` của danh sách có trùng). Mỗi cái có phản ví dụ riêng.

In [20]:
def bad_only_x(points):
    """SAI: chỉ xét x, quên y."""
    xs = {x for x, y in points}
    s = min(xs) + max(xs)
    return all((s - x) in xs for x in xs)


def bad_floor_div(points):
    """SAI: m = (min+max)//2 rồi phản xạ 2m - x (mất nửa đơn vị khi tổng lẻ)."""
    pts = {(x, y) for x, y in points}
    m = (min(x for x, y in points) + max(x for x, y in points)) // 2
    return all((2 * m - x, y) in pts for x, y in points)


def bad_mean(points):
    """SAI: lấy trung bình x của cả danh sách (kể cả trùng) làm trục."""
    pts = {(x, y) for x, y in points}
    s = 2 * sum(x for x, y in points) / len(points)
    return all((s - x, y) in pts for x, y in points)


print("only_x  :", bad_only_x([[1, 1], [-1, -1]]), "(đúng: False)")
print("floor   :", bad_floor_div([[0, 0], [1, 0]]), "(đúng: True)")
print("mean    :", bad_mean([[0, 0], [0, 0], [0, 0], [4, 0]]), "(đúng: False)")
print("mean    :", bad_mean([[0, 0], [1, 0], [1, 0], [1, 0]]), "(đúng: True)")
assert bad_only_x([[1, 1], [-1, -1]]) is True              # sai so với False
assert bad_floor_div([[0, 0], [1, 0]]) is False            # sai so với True
assert bad_mean([[0, 0], [1, 0], [1, 0], [1, 0]]) is False # sai so với True

only_x  : True (đúng: False)
floor   : False (đúng: True)
mean    : False (đúng: False)
mean    : False (đúng: True)


### 12.3. Đối chiếu ngẫu nhiên (có seed) — năm cách + `Solution` + oracle

Sinh bộ điểm theo ba kiểu để có **cả `True` lẫn `False`** nhiều:

1. **Đối xứng thật:** tạo vài điểm nửa trái, phản xạ qua một trục (nguyên hoặc nửa nguyên), thêm vài điểm trên trục, thêm trùng.
2. **Đối xứng bị phá:** như (1) nhưng đổi `y` hoặc xóa một điểm.
3. **Hoàn toàn ngẫu nhiên** trong lưới nhỏ.

Tọa độ trong `[−R, R]`, `R = 6`, `n` từ `1` đến `12`.

In [21]:
def gen_symmetric(rng, R):
    s = rng.randint(-R, R)                                  # s = 2m: trục m nguyên hoặc nửa nguyên
    pts = []
    for _ in range(rng.randint(0, 5)):
        x, y = rng.randint(-R, R), rng.randint(-R, R)
        if abs(s - x) <= R:
            pts += [[x, y], [s - x, y]]
    for _ in range(rng.randint(0, 2)):
        if s % 2 == 0 and abs(s // 2) <= R:
            pts.append([s // 2, rng.randint(-R, R)])         # điểm nằm đúng trên trục
    if not pts:
        pts = [[rng.randint(-R, R), rng.randint(-R, R)]]
    for _ in range(rng.randint(0, 2)):
        pts.append(list(rng.choice(pts)))                    # thêm điểm trùng
    return pts


rng = random.Random(356)
R = 6
count = Counter()
for case in range(4000):
    kind = rng.choice(["sym", "broken", "random"])
    if kind == "sym":
        pts = gen_symmetric(rng, R)
    elif kind == "broken":
        pts = gen_symmetric(rng, R)
        j = rng.randrange(len(pts))
        if rng.random() < 0.5 and len(pts) > 1:
            pts.pop(j)
        else:
            pts[j] = [pts[j][0], max(-R, min(R, pts[j][1] + rng.choice([-1, 1])))]
    else:
        pts = [[rng.randint(-R, R), rng.randint(-R, R)] for _ in range(rng.randint(1, 12))]
    expected = oracle_reflect(pts, R)
    count[expected] += 1
    for f in METHODS:
        assert f(pts) == expected, (f, pts, expected)
print("4000 bộ điểm ngẫu nhiên: mọi cách khớp oracle; phân bố đáp án:", dict(count))
assert count[True] > 300 and count[False] > 300           # test không "tầm thường"

4000 bộ điểm ngẫu nhiên: mọi cách khớp oracle; phân bố đáp án: {True: 2107, False: 1893}


In [22]:
# Ca lớn: n = 10^4, tọa độ tới 10^8: bộ đối xứng thật và bộ bị phá 1 điểm; các cách O(n) / O(n log n) trả lời nhanh
import time
rng = random.Random(3560)
s = 2 * rng.randint(-10**7, 10**7) + 1                      # trục nửa nguyên
half = []
for _ in range(5000):
    x = rng.randint(-5 * 10**7, 5 * 10**7)
    y = rng.randint(-10**8, 10**8)
    half.append([x, y])
big = half + [[s - x, y] for x, y in half]
rng.shuffle(big)
big_broken = [p[:] for p in big]
big_broken[0][1] += 1 if big_broken[0][1] < 10**8 else -1

for name, f in [("minmax", is_reflected_minmax), ("rows", is_reflected_rows), ("sorted_pairs", is_reflected_sorted_pairs)]:
    t0 = time.perf_counter()
    a, b = f(big), f(big_broken)
    print(f"{name:13s}: đối xứng -> {a}, bị phá -> {b}, {time.perf_counter() - t0:.3f}s")
    assert a is True and b is False

minmax       : đối xứng -> True, bị phá -> False, 0.011s
rows         : đối xứng -> True, bị phá -> False, 0.008s


sorted_pairs : đối xứng -> True, bị phá -> False, 0.018s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ xét `x`, quên `y`

**Code sai.**
```python
xs = {x for x, y in points}
s = min(xs) + max(xs)
return all((s - x) in xs for x in xs)
```
**Vì sao sai.** Phản xạ qua đường **thẳng đứng** giữ nguyên `y`. `(1,1)` và `(−1,−1)` có `x` đối xứng nhưng `y` khác nhau, không phải đối xứng.

**Code đúng.** So cả cặp `(x, y)`: `(s − x, y) in point_set`.

In [23]:
print("sai :", bad_only_x([[1, 1], [-1, -1]]))        # True  (sai)
print("đúng:", Solution().isReflected([[1, 1], [-1, -1]]))   # False

sai : True
đúng: False


### Lỗi 2: Chia nguyên `(min + max) // 2`

**Code sai.**
```python
m = (min_x + max_x) // 2
... (2 * m - x, y) in point_set
```
**Vì sao sai.** Khi `min_x + max_x` **lẻ**, trục là nửa nguyên (ví dụ `0.5`); `//` làm tròn xuống thành `0` và phản xạ lệch. Ví dụ `[[0,0],[1,0]]`: đáp án `True` (trục `x = 0.5`).

**Code đúng.** Giữ `s = min_x + max_x` (nguyên), phản xạ bằng `s − x`.

In [24]:
print("sai :", bad_floor_div([[0, 0], [1, 0]]))        # False (sai)
print("đúng:", Solution().isReflected([[0, 0], [1, 0]]))     # True

sai : False
đúng: True


### Lỗi 3: Lấy **trung bình** `x` làm trục

**Code sai.**
```python
m = sum(x for x, y in points) / len(points)
```
**Vì sao sai.** Điểm **lặp** làm trung bình bị kéo lệch. `[[0,0],[1,0],[1,0],[1,0]]`: tập điểm `{(0,0),(1,0)}` đối xứng qua `0.5` (`True`), nhưng trung bình `x = 0.75`. (Nếu không có điểm lặp thì bộ đối xứng quả thật có trung bình `x` bằng trục, nhưng với điểm lặp thì trung bình bị lệch.)

**Code đúng.** Trục là `(min_x + max_x) / 2`, không phụ thuộc số lần lặp.

In [25]:
print("sai :", bad_mean([[0, 0], [1, 0], [1, 0], [1, 0]]))     # False (sai)
print("đúng:", Solution().isReflected([[0, 0], [1, 0], [1, 0], [1, 0]]))   # True

sai : False
đúng: True


### Lỗi 4: Bỏ `list` vào `set` → `TypeError`

**Code sai.**
```python
point_set = set()
for p in points:
    point_set.add(p)         # p là list [x, y]
```
**Vì sao sai.** `list` không băm được (`unhashable type: 'list'`).

**Code đúng.** `point_set.add((x, y))` hoặc `tuple(p)`.

In [26]:
try:
    s = set()
    for p in [[1, 2]]:
        s.add(p)
except TypeError as e:
    print("sai: TypeError:", e)
print("đúng:", {tuple(p) for p in [[1, 2]]})

sai: TypeError: unhashable type: 'list'
đúng: {(1, 2)}


### Lỗi 5: Coi điểm lặp là các điểm riêng biệt (dùng `Counter`)

**Code sai.**
```python
from collections import Counter
cnt = Counter(map(tuple, points))
reflected = Counter((s - x, y) for x, y in points)
return cnt == reflected        # số lần xuất hiện cũng phải khớp
```
**Vì sao sai.** Đề nói "tập điểm ban đầu giống tập điểm sau phản xạ" và chú ý "có thể có điểm lặp" — nghĩa là chỉ quan tâm **có/không** (tập), không phải số lần xuất hiện. `[[0,0],[0,0],[1,0]]`: trục `x = 0.5`; điểm `(0,0)` xuất hiện `2` lần nhưng `(1,0)` chỉ `1` lần; `Counter` khác nhau ⇒ trả `False` sai. (Đối chiếu lại đề gốc nếu bạn có Premium để xác nhận cách hiểu "tập".)

**Code đúng.** Dùng `set`, không `Counter`.

In [27]:
def bad_counter(points):
    s = min(x for x, y in points) + max(x for x, y in points)
    cnt = Counter(map(tuple, points))
    reflected = Counter((s - x, y) for x, y in points)
    return cnt == reflected


print("sai :", bad_counter([[0, 0], [0, 0], [1, 0]]))          # False (sai theo cách hiểu tập)
print("đúng:", Solution().isReflected([[0, 0], [0, 0], [1, 0]]))   # True

sai : False
đúng: True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "đối xứng / phản xạ" ⇒ tìm **bất biến buộc trục**: cặp cực trị (`min`, `max`) phải đối xứng nhau, nên trục **được xác định duy nhất**. Đừng thử mọi trục.
- Dùng `s = min_x + max_x` thay vì `m` để tránh `/ 2` và số thực: phản xạ `(s − x, y)`.
- "Có/không có điểm trong tập" ⇒ `set` các **tuple** `(x, y)`. List không băm được.
- Điểm lặp trong đề tập hợp ⇒ để `set` tự gộp.
- Phản xạ qua đường **thẳng đứng** giữ nguyên `y` ⇒ có thể xử lý **từng hàng ngang** độc lập.
- Nếu không muốn băm: **sắp xếp + hai con trỏ** (kiểm tra cặp đầu–cuối có cùng tổng).
- Với số lớn (`10^8`), tổng `2·10^8` vẫn nằm gọn trong số nguyên Python (không tràn); nếu viết C++/Java thì cũng còn trong `int` 32-bit (`≤ 2·10^8 < 2^31`).
- Kiểm tra nhanh: tự thử `[[0,0],[1,0]]` (trục nửa nguyên), `[[5,7]]` (một điểm), `[[1,1],[-1,-1]]` (khác `y`).

## 15. Biến thể của bài

1. **Trục ngang** (đường thẳng song song trục `x`): hoán đổi `x ↔ y` rồi áp dụng cùng thuật toán.
2. **Trả về trục** (hoặc "không có"): `s / 2` nếu thỏa — ví dụ hàm `find_axis` ở dưới.
3. **Đếm số điểm cần thêm** để bộ điểm trở nên đối xứng qua trục `x = (min_x + max_x)/2` (đếm số điểm `(s − x, y)` còn thiếu).
4. **Đường thẳng bất kỳ** (không nhất thiết song song trục): phức tạp hơn nhiều; có thể thử các trục là trung trực của các cặp điểm, so sánh theo hình học (ngoài phạm vi bài này).
5. **Đối xứng tâm** (quay `180°` quanh một điểm): cùng lập luận cực trị, tâm bị buộc là `((min_x + max_x)/2, (min_y + max_y)/2)`; phép đối xứng là `(sx − x, sy − y)` với `sx = min_x + max_x`, `sy = min_y + max_y` (hàm `is_point_symmetric` bên dưới, có đối chiếu vét cạn).
6. **Bài liên quan:** LeetCode 149 (Max Points on a Line), 593 (Valid Square), 223 (Rectangle Area), 1232 (Check If It Is a Straight Line).

In [28]:
def find_axis(points):
    """Trả về hoành độ trục (float) nếu có, ngược lại None."""
    pts = {(x, y) for x, y in points}
    s = min(x for x, y in points) + max(x for x, y in points)
    return s / 2 if all((s - x, y) in pts for x, y in pts) else None


def min_points_to_add(points):
    """Số điểm (khác nhau) cần thêm để bộ điểm đối xứng qua x = (min_x + max_x)/2."""
    pts = {(x, y) for x, y in points}
    s = min(x for x, y in points) + max(x for x, y in points)
    return sum(1 for x, y in pts if (s - x, y) not in pts)


print(find_axis([[1, 1], [-1, 1]]), find_axis([[0, 0], [1, 0]]), find_axis([[1, 1], [-1, -1]]))
print(min_points_to_add([[1, 1], [-1, -1]]), min_points_to_add([[0, 0], [4, 0], [1, 3]]))
assert find_axis([[1, 1], [-1, 1]]) == 0.0
assert find_axis([[0, 0], [1, 0]]) == 0.5
assert find_axis([[1, 1], [-1, -1]]) is None
assert min_points_to_add([[1, 1], [-1, -1]]) == 2
assert min_points_to_add([[0, 0], [4, 0], [1, 3]]) == 1


def is_point_symmetric(points):
    pts = {(x, y) for x, y in points}
    sx = min(x for x, y in points) + max(x for x, y in points)
    sy = min(y for x, y in points) + max(y for x, y in points)
    return all((sx - x, sy - y) in pts for x, y in pts)


def point_sym_oracle(points, R):
    pts = {(x, y) for x, y in points}
    return any({(sx - x, sy - y) for x, y in pts} == pts
               for sx in range(-2 * R - 2, 2 * R + 3) for sy in range(-2 * R - 2, 2 * R + 3))


rng = random.Random(15)
for _ in range(400):
    pts = [[rng.randint(-3, 3), rng.randint(-3, 3)] for _ in range(rng.randint(1, 6))]
    if rng.random() < 0.5:
        pts += [[-x, -y] for x, y in pts]                       # tạo bộ đối xứng tâm
    assert is_point_symmetric(pts) == point_sym_oracle(pts, 3)
print(is_point_symmetric([[1, 2], [-1, -2]]), is_point_symmetric([[1, 2], [-1, 2], [0, 0]]))

0.0 0.5 None
2 1
True False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Thử mọi cặp `x` | trục = trung điểm của hai `x` bất kỳ | `O(n³)` | `O(n)` | oracle / hiểu đề |
| 1b. Thử trục từ điểm đầu | điểm đầu phải có "bạn" cùng hàng | `O(n²)` | `O(n)` | bước trung gian |
| 3. `min + max` + `set` | trục duy nhất `s = min_x + max_x` | `O(n)` | `O(n)` | **nên nộp** |
| 4. Nhóm theo `y` + hai con trỏ | mỗi hàng đối xứng qua cùng `s` | `O(n log n)` | `O(n)` | không muốn băm tuple |
| 5. Chia trái/phải + sort + ghép cặp | ghép trái-phải theo thứ tự sau sắp | `O(n log n)` | `O(n)` | ngôn ngữ thiếu hash cho cặp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `reflect((3, 8), 10)`, `min(x for x, y in [[2, 1], [-4, 0], [7, 7]]) + max(x for x, y in [[2, 1], [-4, 0], [7, 7]])`, `Solution().isReflected([[2, 5]])`, `Solution().isReflected([[0, 0], [3, 1]])`.

**Bài 2 (tính tay).** Với `points = [[0,0],[6,0],[2,4],[4,4],[3,9]]`: tính `s`, liệt kê điểm phản xạ của từng điểm và kết luận. Sau đó đổi `[4,4]` thành `[4,5]` và kết luận lại.

**Bài 3 (điền TODO).** Hoàn thành `is_reflected_todo(points)` theo Cách 3.

In [29]:
def is_reflected_todo(points):
    # TODO 1: point_set = {(x, y) for x, y in points}
    # TODO 2: s = min(x for x, y in points) + max(x for x, y in points)
    # TODO 3: return all((s - x, y) in point_set for x, y in points)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn code mẫu, viết `is_reflected_rows2(points)` theo Cách 4 (nhóm theo `y`, sắp các `x`, hai con trỏ cùng tổng `s`).

**Bài 5 (biến thể / mini challenge).** Viết `is_reflected_horizontal(points)`: trục **ngang** `y = c` (song song trục `x`) — gợi ý: hoán đổi tọa độ. Rồi viết `reflect_either(points)` trả `"vertical"`, `"horizontal"`, `"both"` hoặc `"none"` tùy bộ điểm có đối xứng qua trục đứng / trục ngang / cả hai / không.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `reflect((3, 8), 10) = (7, 8)`; `-4 + 7 = 3`; một điểm: `True`; `[[0,0],[3,1]]`: khác `y`: `False`. Bài 3: ba dòng đúng theo ba TODO.
- **Mức 2 (vừa):** Bài 2: `s = 0 + 6 = 6`; `(0,0)→(6,0)` ✓, `(6,0)→(0,0)` ✓, `(2,4)→(4,4)` ✓, `(4,4)→(2,4)` ✓, `(3,9)→(3,9)` ✓ ⇒ `True`. Đổi `(4,4)` thành `(4,5)`: `(2,4)→(4,4)` không có ⇒ `False`.
- **Mức 3 (sâu):** Bài 4: gom `rows[y] = set(x)`; lấy `s` từ hàng đầu; mỗi hàng hai con trỏ `i, j` với `i <= j`. Bài 5: đối xứng ngang = áp thuật toán trên các điểm đã đổi `[y, x]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert reflect((3, 8), 10) == (7, 8)
assert min(x for x, y in [[2, 1], [-4, 0], [7, 7]]) + max(x for x, y in [[2, 1], [-4, 0], [7, 7]]) == 3
assert Solution().isReflected([[2, 5]]) is True
assert Solution().isReflected([[0, 0], [3, 1]]) is False

# Bài 2
pts2 = [[0, 0], [6, 0], [2, 4], [4, 4], [3, 9]]
assert min(p[0] for p in pts2) + max(p[0] for p in pts2) == 6
assert Solution().isReflected(pts2) is True
assert Solution().isReflected([[0, 0], [6, 0], [2, 4], [4, 5], [3, 9]]) is False


# Bài 3
def is_reflected_todo_ans(points):
    point_set = {(x, y) for x, y in points}
    s = min(x for x, y in points) + max(x for x, y in points)
    return all((s - x, y) in point_set for x, y in points)


for pts, expected in examples:
    assert is_reflected_todo_ans(pts) == expected


# Bài 4
def is_reflected_rows2(points):
    rows = defaultdict(set)
    for x, y in points:
        rows[y].add(x)
    s = None
    for xs_set in rows.values():
        xs = sorted(xs_set)
        if s is None:
            s = xs[0] + xs[-1]
        i, j = 0, len(xs) - 1
        while i <= j:
            if xs[i] + xs[j] != s:
                return False
            i += 1
            j -= 1
    return True


rng = random.Random(19)
for _ in range(500):
    pts = [[rng.randint(-4, 4), rng.randint(-4, 4)] for _ in range(rng.randint(1, 8))]
    assert is_reflected_rows2(pts) == Solution().isReflected(pts)


# Bài 5
def is_reflected_horizontal(points):
    return Solution().isReflected([[y, x] for x, y in points])


def reflect_either(points):
    v = Solution().isReflected(points)
    h = is_reflected_horizontal(points)
    return "both" if v and h else "vertical" if v else "horizontal" if h else "none"


assert reflect_either([[1, 1], [-1, 1], [0, 5]]) == "vertical"
assert reflect_either([[1, 1], [1, -1], [7, 0]]) == "horizontal"
assert reflect_either([[1, 1], [-1, 1]]) == "both"          # một hàng ngang: mỗi điểm tự đối xứng qua y = 1
assert reflect_either([[1, 1], [-1, 1], [1, -1], [-1, -1]]) == "both"
assert reflect_either([[0, 0], [1, 2], [5, 9]]) == "none"
print("Tất cả đáp án bài tập đều đúng")

Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (`min + max` với `set` các tuple)

```python
class Solution:
    def isReflected(self, points: List[List[int]]) -> bool:
        min_x, max_x = float("inf"), float("-inf")
        point_set = set()
        for x, y in points:
            min_x = min(min_x, x)
            max_x = max(max_x, x)
            point_set.add((x, y))
        s = min_x + max_x                       # = 2 * hoành độ trục
        return all((s - x, y) in point_set for x, y in points)
```

**Checklist**

- [ ] Đã dùng `s = min_x + max_x` (nguyên), không chia `2`/không dùng `//`?
- [ ] Phản xạ giữ nguyên `y` và đổi `x → s − x`; so cả cặp `(x, y)`?
- [ ] Điểm bỏ vào `set` là **tuple**, không phải `list`?
- [ ] Điểm lặp và điểm nằm trên trục được xử lý đúng (set; `s − x = x`)?
- [ ] Không dùng trung bình `x` hay `Counter` (đề hỏi **tập** điểm)?
- [ ] Đã thử: ví dụ đề, `1` điểm, cùng `x`, trục nửa nguyên `[[0,0],[1,0]]`, `y` khác nhau, điểm lặp?
- [ ] Skeleton đúng `class Solution` / `def isReflected(self, points: List[List[int]]) -> bool`?
- [ ] (Premium) đã đối chiếu lại đề gốc nếu có tài khoản?